# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Jessica Amihere
**Student ID:** 57592028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [17]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [18]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content, response.usage
#
# TODO: Call it once with a simple question and print the answer.
answer, usage = ask_llm("who is the president of Ashesi University?")

# TODO: Print response.usage as well — how many tokens did your call consume?
print("Response:")
print(answer)

print("\nToken usage:")
print(usage)

Response:
The president of Ashesi University is Patrick Awuah Jr. He is a Ghanaian educator and entrepreneur who founded Ashesi University in 2002.

Token usage:
CompletionUsage(completion_tokens=32, prompt_tokens=50, total_tokens=82, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008646369, prompt_time=0.004427357, completion_time=0.120980621, total_time=0.125407978)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The system role gives the AI its instructions or sets how it should behave. For example: “Answer in simple language.”
On the other hand, the user role contains what you actually want the AI to do. For example: “Explain artificial intelligence.”

2. A token is a small piece of text that the AI reads or produces. APIs charge per token because a short request uses less processing than a long one, so you pay based on how much text the model handles.

### Part 1.2 — Temperature: the randomness dial

In [19]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
question = "Suggest a name for a savings product for market traders in Accra."

# Temperature 0.0
print("=== Temperature 0.0 ===")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}")

print("\n=== Temperature 1.2 ===")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}")


=== Temperature 0.0 ===
1. ('Here are a few suggestions for a savings product for market traders in Accra:\n\n1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.\n2. **Traders\' Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.\n3. **Accra Market Fund**: This name is straightforward and clearly communicates the product\'s purpose and target audience.\n4. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could encourage market traders to save and collect their earnings.\n5. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.\n6. **Kae Dzi**: "Kae Dzi" is a Ghanaian phrase that means "save for the future". This name could appeal to market traders who are looking to plan for their f

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At 0.0, the answers were more consistent and similar across the five attempts. At 1.2, the suggestions were more varied and creative.
For a loan decision-support system, I would use a low temperature because the system needs consistent and predictable decisions rather than random or creative outputs.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [20]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [21]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    print(f"\n--- V1: {letter_id} ---")
    print(ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    ))

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications factually and neutrally.
Do not invent or assume any details.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    print(f"\n--- V2: {letter_id} ---")
    print(ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0
    ))

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n=== V1 vs V2 Comparison ===")

for letter_id in ["L002", "L006"]:
    v1 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}")
    v2 = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0
    )

    print(f"\n{letter_id}")
    print("V1:", v1)
    print("V2:", v2)



--- V1: L002 ---
("Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects things to improve after the festive season. He doesn't have collateral, but promises to repay the loan as soon as possible.", CompletionUsage(completion_tokens=72, prompt_tokens=133, total_tokens=205, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.00816733, prompt_time=0.006770317, completion_time=0.186541179, total_time=0.193311496))

--- V1: L006 ---
('Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be business-minded and trustworthy, and promises to repay the loan in one year when his businesses are successful.', CompletionUsage(completion_tokens=74, prompt_tokens=135, total_tokens=209, complet

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 was too general, so the response could be less focused or include assumptions. For example, L002’s GHS 25,000 request, lack of collateral, and unclear repayment plan could be missed. V2 fixed this by clearly saying “factual and neutral” and “do not invent or assume any details.”

2. “No invented details” matters because a made-up fact could unfairly affect someone’s loan decision. For example, L006 says Kofi has not started his businesses, so the AI must not call him an experienced business owner. This mistake is called an LLM hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [26]:
import json
import pandas as pd

EXTRACT_PROMPT = """
You are an information extraction assistant for a microfinance loan officer.

Extract information from the loan application and return ONLY a valid JSON object.

The JSON object MUST contain EXACTLY these six keys:
{{
    "applicant_name": "string",
    "amount_ghs": number,
    "purpose": "string",
    "monthly_profit_ghs": number or null,
    "has_collateral_or_guarantor": boolean,
    "repayment_months": number or null
}}

Rules:
1. Use only information explicitly stated in the letter.
2. If a field is not stated in the letter, use null.
3. Do not guess, infer, or invent any information.
4. amount_ghs must be a number, not a string.
5. monthly_profit_ghs must be a number if explicitly stated, otherwise null.
6. repayment_months must be a number if explicitly stated, otherwise null.
7. has_collateral_or_guarantor must be true if the applicant explicitly mentions
   collateral or a guarantor, and false if they explicitly state that they have none.
8. Return ONLY the JSON object. Do not include explanations, comments, or markdown.

Example:

Letter:
"Dear Loan Officer,
My name is Jessica Amihere. I run a small accessory shop and need about GHS 5,000
to purchase more products and rent. My business earns a monthly profit of GHS 700.
My mother will guarantee the loan. I propose to repay the loan over 8 months."

Output:
{{
    "applicant_name": "Jessica Amihere",
    "amount_ghs": 5000,
    "purpose": "purchase more products and rent",
    "monthly_profit_ghs": 700,
    "has_collateral_or_guarantor": true,
    "repayment_months": 8
}}

Now extract the fields from this loan application:

{letter_text}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt,
            temperature=0
        )

        result = response.output_text.strip()

        # Remove Markdown JSON fences if the model returns them
        if result.startswith("```json"):
            result = result[len("```json"):].strip()

        if result.startswith("```"):
            result = result[len("```"):].strip()

        if result.endswith("```"):
            result = result[:-3].strip()

        # Parse the JSON
        extracted = json.loads(result)

        # Check that the required keys are present
        required_keys = {
            "applicant_name",
            "amount_ghs",
            "purpose",
            "monthly_profit_ghs",
            "has_collateral_or_guarantor",
            "repayment_months"
        }

        if set(extracted.keys()) != required_keys:
            print("Warning: JSON does not contain exactly the required keys.")
            return None

        return extracted

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM output as JSON.")
        return None

    except Exception as e:
        print(f"Warning: Extraction failed: {e}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df = pd.DataFrame(results)

# Put letter_id first
columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

df = df[columns]

display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. The example should be separate so the model learns the format, not memorizes or copies information from the six letters.

2. “Use null, do not guess” stops the model from filling missing information with assumptions. Without it, the model may make up values for fields that were not actually mentioned.

3. Temperature 0 is better for extraction because we want the same facts and format every time. For creative tasks, a higher temperature can be better because it gives more varied and original answers.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [27]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.


BRIEF_PROMPT = """
You are assisting a microfinance loan officer.

Review the loan application and the extracted information below.

Provide:
1. Strengths — bullet points based only on the letter.
2. Risks / red flags — bullet points based only on the letter.
3. Missing information the officer should request.
4. Suggested next step — such as "invite for interview", "request documents",
   or "flag for senior review".

Do not approve or reject the application. Final loan decisions must always be
made by human officers. Do not invent or assume information.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""


# TODO: Generate briefs for ALL SIX letters

briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        brief = ask_llm(
            BRIEF_PROMPT.format(
                letter_text=letter_text,
                extracted_json=json.dumps(extracted, indent=2)
            ),
            temperature=0
        )[0]

        briefs[letter_id] = brief



# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
for letter_id in ["L001", "L002", "L006"]:
    print(f"\n{'=' * 50}")
    print(f"LOAN BRIEF — {letter_id}")
    print(f"{'=' * 50}")
    print(briefs[letter_id])



LOAN BRIEF — L001
Here's the review of the loan application:

**Strengths:**
* The applicant has 12 years of experience selling provisions at Makola Market, indicating stability and knowledge of the market.
* The applicant has a steady monthly profit of GHS 900, which suggests a consistent income stream.
* The applicant has saved GHS 2,500 with the susu scheme over two years, demonstrating a ability to save and manage finances.
* The applicant has a guarantor, a teacher, which provides an added layer of security for the loan.

**Risks / red flags:**
* The applicant is expanding into a new area (frozen foods) which may come with unknown risks and challenges.
* The repayment amount of GHS 450 per month is approximately 50% of the applicant's monthly profit, which may be a significant burden.

**Missing information:**
* More details about the deep freezer and the expansion plan, such as the expected increase in profits and the market demand for frozen foods.
* Information about the appli

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Yes. For L003, the system should identify strengths like the GHS 2,800 monthly profit, registered business, fixed deposit, and three apprentices. For L006, it should flag the GHS 50,000 request, no existing business, no collateral, and no clear repayment plan.

2. Practical reason: A human officer needs to check documents and other information before making the final decision.

Ethical reason: An AI could make a wrong judgment that unfairly affects someone's access to money.


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 080de57347949e2430228faa303d5f852e0ac7f1

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [28]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

gold_letters = ["L001", "L003", "L006"]

comparison = []

for field in fields:
    row = {"field": field}
    correct = 0

    for letter_id in gold_letters:
        extracted_value = df.loc[
            df["letter_id"] == letter_id, field
        ].iloc[0]

        gold_value = GOLD[letter_id][field]

        # Case-insensitive comparison for names
        if field == "applicant_name":
            match = str(extracted_value).lower() == str(gold_value).lower()
        else:
            match = extracted_value == gold_value

        row[letter_id] = "✓" if match else "✗"

        if match:
            correct += 1

    row["accuracy"] = f"{correct / 3:.1%}"
    comparison.append(row)

comparison_df = pd.DataFrame(comparison)

display(comparison_df)


,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,100.0%
1,amount_ghs,✓,✓,✓,100.0%
2,purpose,✗,✗,✗,0.0%
3,monthly_profit_ghs,✓,✓,✗,66.7%
4,has_collateral_or_guarantor,✓,✓,✓,100.0%
5,repayment_months,✓,✓,✓,100.0%


### Part 4.2 — Reliability: is the system consistent?

In [29]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

def extract_fields(letter_text, temperature=0):
    response, usage = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        temperature=temperature
    )

    response = response.strip()

    if response.startswith("```json"):
        response = response[len("```json"):].strip()

    if response.endswith("```"):
        response = response[:-3].strip()

    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print("Warning: Could not parse JSON.")
        return None


for temperature in [0, 1.0]:
    results = []

    for i in range(5):
        result = extract_fields(LETTERS["L004"], temperature=temperature)
        results.append(result)

    valid_results = [r for r in results if r is not None]

    unique_results = set(
        json.dumps(r, sort_keys=True)
        for r in valid_results
    )

    print(f"\nTemperature = {temperature}")
    print(f"Valid JSON: {len(valid_results)}/5")
    print(f"Identical across all valid runs: {len(unique_results) == 1}")
    print(f"Unique valid outputs: {len(unique_results)}")



Temperature = 0
Valid JSON: 5/5
Identical across all valid runs: True
Unique valid outputs: 1

Temperature = 1.0
Valid JSON: 5/5
Identical across all valid runs: True
Unique valid outputs: 1


### Part 4.3 — Hallucination probing

In [30]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

test1_prompt = f"""
{SUMMARY_PROMPT_V2}

Also answer this question:
What is the applicant's credit score?

Loan application:
{LETTERS["L002"]}
"""

test1_output, _ = ask_llm(
    test1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)

print("===== TEST 1: Missing Information =====")
print(test1_output)
print("\nPASS if the model says the credit score is not provided.")
print("FAIL if the model invents a credit score.")


# ---------------- TEST 2: Irrelevant information ----------------
# Feed the extractor an irrelevant weather report.

weather_report = """
The weather report for Accra says that the morning will be sunny with
temperatures around 29 degrees Celsius. There may be light rain in the evening.
"""

test2_response, _ = ask_llm(
    EXTRACT_PROMPT.format(letter_text=weather_report),
    temperature=0
)

print("\n===== TEST 2: Irrelevant Input =====")
print(test2_response)
print("\nPASS if the model returns null for the loan-related fields.")
print("FAIL if the model invents an applicant or loan information.")

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
# "Test 1 -- PASS"
# "The applicant's credit score is not provided"

# "Test 2 -- PASS"
# "{
#     "applicant_name": null,
#     "amount_ghs": null,
#     "purpose": null,
#     "monthly_profit_ghs": null,
#     "has_collateral_or_guarantor": null,
#     "repayment_months": null
# }"


===== TEST 1: Missing Information =====
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow but expects it to improve after the festive season. The applicant does not have collateral to offer at the moment. The applicant's credit score is not mentioned in the loan application.

PASS if the model says the credit score is not provided.
FAIL if the model invents a credit score.

===== TEST 2: Irrelevant Input =====
{
    "applicant_name": null,
    "amount_ghs": null,
    "purpose": null,
    "monthly_profit_ghs": null,
    "has_collateral_or_guarantor": null,
    "repayment_months": null
}

PASS if the model returns null for the loan-related fields.
FAIL if the model invents an applicant or loan information.


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. My extraction accuracy was 100% for applicant name, amount_ghs, has_collateral_or_guarantor, repayment_months, 66.7% for monthly_profit_ghs and 0.0% for purpose. The purpose field was the hardest because the model could phrase the same purpose differently from the gold-standard wording.

2. The reliability test showed that temperature 0 gave more consistent results across repeated runs. For production loan systems, low temperature is better because the same information should produce the same output.

3. No, the system did not hallucinate in these tests. In Test 1, it correctly said the credit score was not mentioned. In Test 2, it returned null for the missing loan information instead of creating a fake applicant. Instructions like “do not guess” and checking the LLM's output with code can reduce this risk further.


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. Someone with a good business but poor English could be treated unfairly. The AI might think their application is weak just because they did not write well, even though their business is doing well.

2. Loan letters contain private information. Sending them to another company in another country means that company may see people's personal and financial details. Before using it, the bank should check how the company protects, stores, and uses the data and whether it follows Ghana's data protection laws.

3. a. Human check: A real loan officer must review the AI's work before making a decision.
b. Right to appeal:  Applicants should be able to question or challenge a decision they believe is unfair.


---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.